#### Get Capacity Metrics (Timepoints)
by Capacity by Day

### ! IMPORTANT: check the "Check the timepoint dates" queries before running the entire notebook ! <BR>
Each timepoint takes around 7 seconds, so for an entire days worth of 2880 timepoints this will take 5-6 hours 



In [ ]:
import sempy.fabric as fabric
from datetime import datetime, timedelta
import datetime as dt
import pyspark.sql.functions as f
from delta.tables import *
from pyspark.sql.functions import lit, current_timestamp
import time

In [ ]:
## Parameters

# Input strings for evaluating timepoints
start_date_str = "05 May 2026 21:32"
end_date_str = "11 May 2026 09:30"

# capacity metrics workspace and semantic model
metric_workspace = "ecbf6b7e-09f2-44bc-8a06-17520a34dd52"
metric_dataset = "397ef770-86eb-4949-9eb0-2f73e64bfca5"

#target_capacity_id = "DD2FBC11-4406-4C20-B235-1C0473A477BB"
target_capacity_id = "19E3009C-C803-46CF-BFA7-1E0FD0C6E4E7"

#verbose mode displaying data
display_data = False


In [ ]:
## Variables
#timepoints source
timepoint_gold_lakehouse = "Metrics_Lakehouse"
timepoint_gold_table = "capacity_calendar_timepoints"

#target
silver_table_name = "Metrics_Staging_Lakehouse.timepoint_metrics_silver"
gold_lakehouse = "Metrics_Lakehouse"
gold_table_name = "timepoint_metrics"

count_of_connectivity_errors = 0

##### Check the timepoint dates

No point running the time range if the calendar timepoints are not there in the timepoint table

In [ ]:
query = f"""
    SELECT CapacityId, date(TimePoint) as TPDate, min(TimePoint) as MinTimepoint,max(TimePoint) as MaxTimepoint
    FROM {timepoint_gold_lakehouse}.{timepoint_gold_table}
    where CapacityId = "{target_capacity_id}"
    group by CapacityId, date(TimePoint)
    order by CapacityId, TPDate
    """
check_dates = spark.sql(query)
display(check_dates)

##### And check the timepoint dates in the target

This is a long running process, so avoid refetching data already there

In [ ]:
query = f"""
    SELECT CapacityId, date(TimePoint) as TPDate, min(TimePoint) as MinTimepoint,max(TimePoint) as MaxTimepoint
    FROM {gold_lakehouse}.{gold_table_name}
    where CapacityId = "{target_capacity_id}"   
    group by CapacityId,date(TimePoint)
    order by CapacityId, TPDate desc
    """
check_dates_metrics = spark.sql(query)
display(check_dates_metrics)

In [ ]:
# clean Silver table
try:
    query = "DELETE FROM " + silver_table_name
    spark.sql(query)

except Exception as ex:
    print("INFO: Silver table doesn't exist yet.")

## Function for fetching timepoint detail based on input capacity and timepoint

In [ ]:
def get_detailed_data(capacity_id, timepoint):

    capacity_id = capacity_id.upper()
    date_obj = timepoint
    date_part = f"DATE({date_obj.year}, {date_obj.month}, {date_obj.day})"
    time_part = f"TIME({date_obj.hour}, {date_obj.minute}, {date_obj.second})"
    timepoint_expr = f"{date_part} + {time_part}"

    dax_query = f"""
    DEFINE
        MPARAMETER 'TimePoint' = 
            ({timepoint_expr})

        MPARAMETER 'CapacitiesList' = 
            {{"{capacity_id}"}}

        VAR __DS0FilterTable = 
            TREATAS(
                {{"'Timepoint Background Detail'[Billing type]",
                    "'Timepoint Background Detail'[Operation Id]",
                    "'Items'[Virtualised workspace]",
                    "'Timepoint Background Detail'[Start]",
                    "'Timepoint Background Detail'[End]"}},
                'Timepoint detail page optional columns (background operations)'[DynamicColumnsTimepointBackgroundOperations Fields]
            )
        VAR __DS0FilterTable3 = 
            TREATAS({{"{capacity_id}"}}, 'Capacities'[Capacity Id])

        VAR __DS0FilterTable4 = 
            TREATAS({{{date_part}}}, 'Dates'[Date])

        VAR __DS0Core = 
            SUMMARIZECOLUMNS(
                ROLLUPADDISSUBTOTAL(
                    ROLLUPGROUP(
                        'Timepoint Background Detail'[Operation start time],
                        'Timepoint Background Detail'[Operation end time],
                        'Timepoint Background Detail'[Status],
                        'Timepoint Background Detail'[Operation],
                        'Timepoint Background Detail'[User],
                        'Items'[Workspace name],
                        'Items'[Item kind],
                        'Items'[Unique key],
                        'Timepoint Background Detail'[Billing type],
                        'Timepoint Background Detail'[Operation Id],
                        'Items'[Virtualised workspace],
                        'Timepoint Background Detail'[Start],
                        'Timepoint Background Detail'[End],
                        'Items'[Item name]
                    ), "IsGrandTotalRowTotal"
                ),
                __DS0FilterTable,
                __DS0FilterTable3,
                __DS0FilterTable4,
                "SumDuration__s_", CALCULATE(SUM('Timepoint Background Detail'[Duration (s)])),
                "SumTotal_CU__s_", CALCULATE(SUM('Timepoint Background Detail'[Total CU (s)]))
            )
    EVALUATE
        __DS0Core

    """

    #print(dax_query)
    start_time = time.time()
    timepoint_summary = fabric.evaluate_dax(
        workspace=metric_workspace,
        dataset=metric_dataset,
        dax_string=dax_query
    )
    end_time = time.time()

    if display_data:
        display(timepoint_summary)
    
    timepoint_summary.columns = ['Operation_start_time','Operation_end_time','Status','Operation','User','Workspace_name'
        ,'Item_kind','Unique_key','Billing_type','Operation_Id','Virtualised_workspace','Start','End','Item_name','IsGrandTotalRowTotal'
        ,'SumDuration_s','SumTotal_CU_s']

    row_count = timepoint_summary['Operation_start_time'].count()
    
    if row_count < 2:
        print(f"INSUFFICIENT ROWS: Elapsed time: {end_time - start_time:.4f} seconds. {row_count} row(s) captured")
        return
    
    timepoint_summary = spark.createDataFrame(timepoint_summary)
    row_count = timepoint_summary.count()
    print(f"Elapsed time: {end_time - start_time:.4f} seconds. {row_count} row(s) captured")

    timepoint_summary = timepoint_summary.withColumn("Timepoint",lit(date_obj)).withColumn("CapacityID",lit(capacity_id)).withColumn("insert_date",current_timestamp())
    timepoint_summary.write.mode("append").option("mergeSchema", "true").format("delta").saveAsTable(silver_table_name)


In [ ]:
# Query capacity metrics and populate df DataFrame
query = f"""
    SELECT CapacityId, TimePoint
    FROM {timepoint_gold_lakehouse}.{timepoint_gold_table}
    WHERE TimePoint BETWEEN to_timestamp('{start_date_str}', 'dd MMMM yyyy HH:mm')
                        AND to_timestamp('{end_date_str}', 'dd MMMM yyyy HH:mm')
                        and CapacityId = "{target_capacity_id}"
    ORDER BY CapacityId, TimePoint DESC
    """
df = spark.sql(query)

# Loop through distinct timepoints and print each one
timepoints = (
    df.select("CapacityId", "TimePoint")
      .distinct()
      .orderBy("CapacityId", "TimePoint")
      .collect()
)

if display_data:
    display(timepoints)

for row in timepoints:
    print(row["CapacityId"], row["TimePoint"])
    get_detailed_data(row["CapacityId"], row["TimePoint"])


In [ ]:
query = f"SELECT * FROM {silver_table_name} where Unique_key is not NULL" 
silver_df = spark.sql(query)
if display_data:
    display(silver_df)

In [ ]:
silver_count = silver_df.count()
display(silver_count)

In [ ]:
if display_data:
        print(f"Getting read to merge from {silver_table_name} to {gold_lakehouse}.{gold_table_name}")

In [ ]:
# Check if gold table exists
if spark._jsparkSession.catalog().tableExists(gold_lakehouse, gold_table_name):
    # if exists -> MERGE to gold
    print("INFO: Gold table exists and will be merged.")

    table_full_name = f"{gold_lakehouse}.{gold_table_name}"
    gold_df = DeltaTable.forName(spark, table_full_name)
    # Merge silver (s = source) to gold (t = target)
    gold_df.alias('t') \
    .merge(
        silver_df.alias('s'),
        "s.CapacityId = t.CapacityId AND s.TimePoint = t.TimePoint and s.Unique_key = t.Unique_key and s.Operation = t.Operation and s.Operation_Id = t.Operation_Id"
    ) \
    .whenNotMatchedInsert(values =
        {
             "Operation_start_time" : "s.Operation_start_time"
            ,"Operation_end_time" : "s.Operation_end_time"
            ,"Status" : "s.Status"
            ,"Operation" : "s.Operation"
            ,"User" : "s.User"
            ,"Workspace_name" : "s.Workspace_name"
            ,"Item_kind" : "s.Item_kind"
            ,"Unique_key" : "s.Unique_key"
            ,"Billing_type" : "s.Billing_type"
            ,"Operation_Id" : "s.Operation_Id"
            ,"Virtualised_workspace" : "s.Virtualised_workspace"
            ,"Start" : "s.Start"
            ,"End" : "s.End"
            ,"Item_name" : "s.Item_name"
            ,"IsGrandTotalRowTotal" : "s.IsGrandTotalRowTotal"
            ,"SumDuration_s" : "s.SumDuration_s"
            ,"SumTotal_CU_s" : "s.SumTotal_CU_s"
            ,"Timepoint" : "s.Timepoint"
            ,"CapacityID" : "s.CapacityID"
            ,"insert_date" : "s.insert_date"
        }
    ) \
    .execute()

else:
    # else -> INSERT to gold
    print("INFO: Gold table will be created.")
    table_name = f"{gold_lakehouse}.{gold_table_name}"
    silver_df.write.mode("append").option("mergeSchema", "true").format("delta").saveAsTable(table_name)